# Three-SMU live scan

This notebook uses the same audited generator as the CLI. It does not import QCoDeS directly. Real connection and writes require manually changing `AUTHORIZE_WRITES` after reviewing both local TOML files.

In [ ]:
from pathlib import Path
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

from attodry_control.three_smu import ThreeSmuSession
from attodry_control.three_smu_config import (
    load_three_smu_hardware, load_three_smu_scan, validate_plan_targets,
)


In [ ]:
HARDWARE_TOML = Path('../config/three_smu_hardware.local.toml')
PLAN_TOML = Path('../config/three_smu_scan.local.toml')
OUTPUT_DIR = Path('../run_data/three_smu')
AUTHORIZE_WRITES = False

hardware = load_three_smu_hardware(HARDWARE_TOML)
plan = load_three_smu_scan(PLAN_TOML)
points = validate_plan_targets(hardware, plan)
print(f'Validated {len(points)} points without opening hardware.')


In [ ]:
elapsed, bias_current = [], []
with ThreeSmuSession.open(
    hardware, plan, authorize_writes=AUTHORIZE_WRITES,
) as session:
    for sample in session.run(output_dir=OUTPUT_DIR):
        elapsed.append(sample.elapsed_s)
        bias_current.append(sample.readings['smu_bias'].reading.current_a)
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6.4, 4.0), constrained_layout=True)
        ax.plot(elapsed, bias_current, marker='o')
        ax.set(xlabel='Elapsed time (s)', ylabel='Bias current (A)', title='Live Three-SMU scan')
        ax.grid(True, alpha=0.25)
        display(fig)
        plt.close(fig)
print(f'Run directory: {session.last_run_dir}')
